<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Munsell_10B_9_2_roundtrip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Munsell → CIE Lab → Munsell round trip

This notebook converts the Munsell specification **10B 9/2** to CIE 1931 xyY, CIE 1931 XYZ, and CIE L*a*b*, then reconstructs a Munsell specification from the resulting Lab triplet.

The calculation uses the `colour-science` implementation of ASTM D1535 / Munsell renotation interpolation. Lab is calculated relative to the CIE 1931 2° D65 reference white. The reconstructed specification is generally approximate because the Munsell system is discretised and the inverse transform involves interpolation.

In [1]:
# Install in Google Colab or another clean Jupyter environment.
!pip -q install colour-science

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.5 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import colour

np.set_printoptions(precision=6, suppress=True)
print('colour-science version:', colour.__version__)

colour-science version: 0.4.7


In [3]:
# Input Munsell notation. Change this value to process another chromatic Munsell code.
munsell_input = '10B 9/2'

# Use a standard notation form accepted by colour-science.
munsell_input = re.sub(r'\s+', ' ', munsell_input.strip().upper())
print('Input Munsell code:', munsell_input)

Input Munsell code: 10B 9/2


In [4]:
# Forward conversion: Munsell -> CIE xyY -> CIE XYZ -> CIE Lab.
# xyY and XYZ are represented on the usual 0-1 Y scale in colour-science.
xyY = colour.munsell_colour_to_xyY(munsell_input)
XYZ = colour.xyY_to_XYZ(xyY)
Lab = colour.XYZ_to_Lab(XYZ, illuminant=colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65'])

print(f'Munsell : {munsell_input}')
print(f'CIE xyY: x = {xyY[0]:.6f}, y = {xyY[1]:.6f}, Y = {xyY[2]:.6f}')
print(f'CIE XYZ (0-1): X = {XYZ[0]:.6f}, Y = {XYZ[1]:.6f}, Z = {XYZ[2]:.6f}')
print(f'CIE Lab (D65/2°): L* = {Lab[0]:.4f}, a* = {Lab[1]:.4f}, b* = {Lab[2]:.4f}')

Munsell : 10B 9/2
CIE xyY: x = 0.294900, y = 0.307600, Y = 0.766956
CIE XYZ (0-1): X = 0.735290, Y = 0.766956, Z = 0.991108
CIE Lab (D65/2°): L* = 90.1813, a* = 1.3215, b* = -10.7435


In [5]:
# Reverse conversion: Lab -> XYZ -> xyY -> Munsell.
# Reusing the same D65/2° white point makes the transform self-consistent.
XYZ_roundtrip = colour.Lab_to_XYZ(Lab, illuminant=colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65'])
xyY_roundtrip = colour.XYZ_to_xyY(XYZ_roundtrip)
munsell_roundtrip = colour.xyY_to_munsell_colour(xyY_roundtrip)

print('Lab triplet used for reverse conversion:', np.round(Lab, 4))
print('Recovered CIE xyY:', np.round(xyY_roundtrip, 6))
print('Recovered Munsell code:', munsell_roundtrip)

Lab triplet used for reverse conversion: [ 90.1813   1.3215 -10.7435]
Recovered CIE xyY: [ 0.2949    0.3076    0.766956]
Recovered Munsell code: 0.0PB 9.0/2.0


In [6]:
# Numerical round-trip diagnostics.
delta_XYZ = XYZ_roundtrip - XYZ
delta_xyY = xyY_roundtrip - xyY

print('ΔXYZ:', delta_XYZ)
print('ΔxyY:', delta_xyY)
print('Maximum absolute XYZ difference:', np.max(np.abs(delta_XYZ)))

# Note: a recovered Munsell notation may contain decimal hue/value/chroma terms.
# That is expected for an inverse interpolation, even when the numerical XYZ round trip is near machine precision.

ΔXYZ: [ 0.  0.  0.]
ΔxyY: [ 0.  0.  0.]
Maximum absolute XYZ difference: 1.11022302463e-16


## Interpretation

- `Lab` is a D65/2° CIE L*a*b* coordinate, calculated from the Munsell-derived XYZ values.
- The return value of `xyY_to_munsell_colour` is an interpolated Munsell notation, so it can be expressed with decimals rather than exactly `10B 9/2`.
- This is a computational colour-coordinate conversion, not a physical spectral reconstruction. For instrument measurements, specify the illuminant, standard observer, geometry, and any required chromatic adaptation consistently.
